# 9b · Which gap rule? — every case, measured

Notebook 9 produces TextGrids. **This one asks whether you should believe them.**

CTC is peaky. The model emits one confident frame per phone and blanks in
between, so on the shipped example the labelled frames cover a **median 32 %** of
the timeline (10–75 % across 55 segments). Everything else in a published
TextGrid is a *rule* deciding who gets the blank frames — which means the rule
matters more than the model does.

There are seven such rules in `kolsch_align.py`. This notebook runs all of them
on your own data and measures the four things that actually distinguish them:

| § | question | needs a reference? |
|---|---|---|
| 1 | How much of the TextGrid is measured, and how much is invented? | no |
| 2 | Where does each rule put the blank run? | no |
| 3 | Do phones swallow the pauses between words? | no |
| 4 | How far are the boundaries from another system's? | **yes** |

**§1–3 run on the shipped example.** §4 needs reference TextGrids you supply —
from MFA, MAUS, or by hand — and is skipped cleanly if you have none.

> **The honest summary, before the numbers.** `vc` is the default. Over 941 word
> boundaries on 84 field recordings `vc-sil` beats it against both MFA and MAUS,
> but against the only human-placed boundaries in this project `vc` is 7.2 ms out
> and `vc-sil` is 62.0. Three boundaries is an anecdote; 941 against two systems
> that share an HMM-GMM lineage is not ground truth. They disagree, and nothing
> in this notebook settles it — it shows you the trade so you can pick knowing
> what you are picking.


In [ ]:
!pip -q install torch torchaudio transformers librosa soundfile pandas matplotlib
import numpy as np, pandas as pd, torch
print("torch", torch.__version__, "·",
      "cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# === Portable setup — identical paths in VS Code, Jupyter & Google Colab ===
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    _here = any((Path(p)/"kolsch_paths.py").exists() for p in [Path.cwd(), *Path.cwd().parents])
    if not _here and not Path("/content/kolsch-tandem/kolsch_paths.py").exists():
        os.system("git clone -q https://github.com/chemvatho/Koelsch-Phoneme-Recognition.git /content/kolsch-tandem")
except Exception:
    pass

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"kolsch_paths.py").exists()),
             Path("/content/kolsch-tandem"))
sys.path.insert(0, str(_root))
from kolsch_paths import ROOT, DATA, SEG, INDEX, LEXICON, MODELS
os.chdir(ROOT)
print("repo root:", ROOT)


In [ ]:
from kolsch_align import (Aligner, MODES, coverage, silence_ms,
                          word_positions, boundary_errors, read_textgrid_tier,
                          write_textgrid, VOCALIC)

MODEL_DIR     = os.environ.get("KOLSCH_MODEL",     os.path.join(MODELS, "kolsch_wav2vec2_model_all"))
PROCESSOR_DIR = os.environ.get("KOLSCH_PROCESSOR", MODEL_DIR)

al = Aligner(MODEL_DIR, PROCESSOR_DIR, lexicon=LEXICON)
print(f"{len(al.vocab)} symbols · blank {al.blank_id} · {al.device}")
print("modes:", ", ".join(MODES))

MANIFEST = os.path.join(SEG, "manifest.csv")
assert os.path.exists(MANIFEST), "run notebook 3 first"
man = pd.read_csv(MANIFEST)
print(len(man), "segments")


## 1 · How much of this TextGrid is actually measured?

`--absorb none` keeps only the frames the model labelled and leaves the rest as
holes. The coverage figure below is the fraction of the timeline that is
**evidence**. Everything above it is rule.


In [ ]:
cov, durs = [], []
for _, r in man.iterrows():
    ph, _, dur = al.align(r["audio_path"], r["text"], mode="none")
    cov.append(coverage(ph, dur)); durs.append(dur)
cov = np.array(cov)
print(f"CTC labels a median {100*np.median(cov):5.1f}% of the timeline "
      f"(range {100*cov.min():.0f}–{100*cov.max():.0f}%, n={len(cov)})")
print(f"So a median {100*(1-np.median(cov)):5.1f}% of every phone duration you "
      f"publish is decided by the gap rule, not by the model.")


## 2 · Where does each rule put the blank run?

One utterance, seven rules, same phone chain. Any duration that moves is the rule
moving it. `vc` should lengthen vowels at the expense of the consonants beside
them — that is the whole point of it.


In [ ]:
row = man.iloc[0]
print(row["text"], "\n")

runs = {m: al.align(row["audio_path"], row["text"], mode=m)[0] for m in MODES}
tbl = pd.DataFrame({"phone": [p["label"] for p in runs["vc"]]})
tbl["class"] = ["vowel" if p in VOCALIC else "cons." for p in tbl["phone"]]
for m in MODES:
    tbl[m] = [round((p["end"] - p["start"]) * 1000) for p in runs[m]]
display(tbl)


In [ ]:
print("mean duration change vs hybrid, ms — the sign is the claim\n")
print(f"{'mode':10}{'vowels':>10}{'consonants':>13}")
for m in ("vc", "vc-onset", "vc-sil"):
    d = tbl[m] - tbl["hybrid"]
    v, c = d[tbl["class"] == "vowel"], d[tbl["class"] == "cons."]
    print(f"{m:10}{v.mean():+9.1f}{c.mean():+13.1f}")
print("\nVowels should gain and consonants lose. If not, the phone-class table")
print("in kolsch_align.py does not match your inventory.")


## 3 · Do the phones swallow the pauses?

**This is the case that is easy to miss.** `vc` has no way to say *silence* — its
phones tile the signal end to end, so a pause between two words ends up inside
whichever phone borders it. `vc-onset` cuts once at the end of the pause, which
fixes the next word's onset and leaves the pause inside the **previous** phone.
Only `vc-sil` emits the pause as a hole, which is what MFA and MAUS do.

On the reference recording the final schwa of *ʃnaɪə* came out **800 ms** under
`vc-onset` against MFA's 160, because the 620 ms MFA labels as silence had
nowhere else to go.


In [ ]:
rows = []
for _, r in man.iterrows():
    rec = {"id": Path(r["audio_path"]).stem}
    for m in ("vc", "vc-onset", "vc-sil"):
        ph, _, _ = al.align(r["audio_path"], r["text"], mode=m)
        rec[m] = silence_ms(ph)
    rows.append(rec)
sil = pd.DataFrame(rows)

print("silence emitted, ms per segment")
print(sil[["vc", "vc-onset", "vc-sil"]].describe().loc[["mean", "50%", "max"]].round(0))
n = int((sil["vc-sil"] > 0).sum())
print(f"\nvc-sil found a pause in {n}/{len(sil)} segments "
      f"(median over those {sil.loc[sil['vc-sil']>0,'vc-sil'].median():.0f} ms).")
print("vc and vc-onset emit zero by construction — every one of those "
      "milliseconds is inside a phone.")


In [ ]:
# The segment where it matters most, drawn.
import matplotlib.pyplot as plt
import librosa

k = int(sil["vc-sil"].idxmax())
r = man.iloc[k]
print(f"{sil.loc[k,'id']}: vc-sil emits {sil.loc[k,'vc-sil']:.0f} ms of silence")
print(r["text"])

wav, sr = librosa.load(r["audio_path"], sr=16000)
fig, ax = plt.subplots(figsize=(13, 3.4))
ax.plot(np.arange(len(wav)) / sr, wav / (np.abs(wav).max() or 1) * 0.4 + 0.7,
        lw=0.4, color="#c8c7c2")
for i, (m, col) in enumerate((("vc", "#2a78d6"), ("vc-onset", "#9dc2ea"),
                              ("vc-sil", "#0d9488"))):
    ph, _, dur = al.align(r["audio_path"], r["text"], mode=m)
    y = -0.25 - i * 0.42
    for p in ph:
        ax.add_patch(plt.Rectangle((p["start"], y), p["end"] - p["start"], 0.34,
                                   facecolor=col, alpha=0.25, edgecolor=col, lw=0.8))
        ax.annotate(p["label"], ((p["start"] + p["end"]) / 2, y + 0.17),
                    ha="center", va="center", fontsize=8)
    ax.annotate(m, (-0.006, y + 0.17), xycoords=("axes fraction", "data"),
                ha="right", va="center", fontsize=9, fontweight="bold", color=col)
ax.set_xlim(0, dur); ax.set_ylim(-1.45, 1.15); ax.set_yticks([])
ax.set_xlabel("time (s)")
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.set_title("gaps in the vc-sil row are pauses; the other two rows have none",
             loc="left", fontsize=10)
plt.tight_layout(); plt.show()


## 4 · Against a reference — optional

Everything above is reference-free. To compare boundaries you need another
system's TextGrids for the **same audio and the same phone chain**, index for
index — otherwise you are comparing two pronunciation dictionaries, not two
aligners.

Drop them in `data/reference_textgrids/<segment-id>.TextGrid` (a `phones` tier)
and this section activates. Produce them with MFA locally, or MAUS at BAS —
**MAUS uploads your audio**, so use it only on recordings you may share.

```bash
mfa align data/segments/ german_mfa german_mfa data/reference_textgrids/
```


In [ ]:
REF_DIR = os.path.join(DATA, "reference_textgrids")
refs = {Path(p).stem: p for p in Path(REF_DIR).glob("*.TextGrid")} \
    if os.path.isdir(REF_DIR) else {}

if not refs:
    print(f"No reference TextGrids in {REF_DIR} — section 4 skipped.")
    print("Everything above stands without one; only this comparison needs it.")
else:
    acc = {}
    for _, r in man.iterrows():
        stem = Path(r["audio_path"]).stem
        if stem not in refs:
            continue
        ref = read_textgrid_tier(refs[stem], "phones")
        chain, _ = al.text_to_chain(r["text"])
        for m in ("hybrid", "vc", "vc-onset", "vc-sil"):
            ph, _, _ = al.align(r["audio_path"], r["text"], mode=m)
            if len(ph) != len(ref):
                continue          # different chain: not comparable, skip loudly
            for pos, val in boundary_errors(ph, ref, chain).items():
                acc.setdefault((m, pos), []).append(val)
    if not acc:
        print("Reference TextGrids found, but none matched the phone chain "
              "index-for-index — check they were aligned on the same transcript.")
    else:
        out = pd.DataFrame(
            [{"mode": m, "position": p, "median |Δ| ms": round(np.median(v), 1),
              "n": len(v)} for (m, p), v in acc.items()])
        display(out.pivot(index="position", columns="mode",
                          values="median |Δ| ms"))
        print("\nAgreement, not accuracy: this says how far apart two systems "
              "place the same boundary, not which one is right.")


## Choosing

| you want | use | why |
|---|---|---|
| the published default, reproduces every figure | `vc` | word onsets untouched, so cross-system comparisons stay valid |
| a TextGrid for **Praat, ELAN or TTS training** | **`vc-sil`** | phones do not span pauses; silence is silence |
| to see what the model actually emitted | `none` | no rule at all — the holes are the honest picture |
| the pre-2026 behaviour | `hybrid` | kept so old reports reproduce byte-for-byte |

`vc-onset` exists as the intermediate step and is **superseded by `vc-sil`**,
which beat it on every column of the 941-boundary comparison. There is no reason
to prefer it.

### The unresolved part

Against MFA and MAUS over 941 word boundaries, `vc-sil` wins. Against the three
human-placed edges in this project, `vc` wins by a lot. The onset-aware modes
move toward MFA and MAUS — which share an HMM-GMM lineage and both put a boundary
at the edge of a silence interval — and away from the person.

Nobody can tell from these numbers how much of that is accuracy and how much is
conformity. **A hand-corrected reference on even 20 utterances would settle it**,
and that is the top item on the project roadmap.


In [ ]:
# Export in whichever mode you chose.
ABSORB = "vc"                                   # or "vc-sil"
TG_DIR = os.path.join(DATA, "textgrids"); os.makedirs(TG_DIR, exist_ok=True)

ok = 0
for _, r in man.iterrows():
    stem = Path(r["audio_path"]).stem
    try:
        ph, wd, dur = al.align(r["audio_path"], r["text"], mode=ABSORB)
        write_textgrid(os.path.join(TG_DIR, f"{stem}.TextGrid"), dur,
                       [("words", wd), ("phones", ph)])
        ok += 1
    except Exception as e:
        print(f"  {stem}: {type(e).__name__}: {e}")
print(f"{ok} TextGrids in mode {ABSORB!r} -> {TG_DIR}")
